In [26]:

from pathlib import Path

import torchvision.models as models
import torch
from torch.utils.tensorboard import SummaryWriter
from xautodl.config_utils import dict2config
from xautodl.models import get_cell_based_tiny_net
import json
from tqdm import tqdm
import logging
import pandas as pd
from kd_vs_hpo.common.utils import load_checkpoint, load_checkpoint_by_idx, get_log_data, get_load_checkpoint_kwargs, load_checkpoint_kwargs_from_module
from kd_vs_hpo.common.analyze import group_confidence_interval
from kd_vs_hpo.common.dataloader import build_cifar10_dataloaders
from kd_vs_hpo.common.analyze import results_to_dataframe
from kd_vs_hpo.common.analyze import analyze_checkpoint_dir, summarize


from kd_vs_hpo.common.train_modules import KDLightningModule 
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from scipy import stats
from omegaconf import DictConfig

logging.basicConfig(level=logging.WARNING)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = analyze_checkpoint_dir('experiments/checkpoints/base_check_3_arch', device=device)
summary = summarize(results)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'kd_loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['kd_loss'])`.


In [4]:
base_exp = results_to_dataframe(summary)

# base architectures

In [129]:
base_res = group_confidence_interval(base_exp.groupby('arch_idx').test_acc)

# kd 1 teacher different seeds

In [107]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = analyze_checkpoint_dir('checkpoints/kd_one_teacher_different_seeds', device=device)
summary = summarize(results)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [108]:
def func(x):
    try:
        x = json.loads(x.params.kd.teachers_mapping.values[0])
    except:
        x = []
    x = sorted(int(i['idx']) for i in x)
    
    return ' '.join(map(str, x))

In [109]:
kd_1_exp = results_to_dataframe(summary)
kd_1_exp['teacher_names'] = kd_1_exp.apply(lambda x: func(x), axis=1)

In [131]:
kd1_res = group_confidence_interval(kd_1_exp[kd_1_exp.role == 'student'].groupby(['arch_idx', 'teacher_names']).test_acc)

# kd 2 teachers different seeds

In [111]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results = analyze_checkpoint_dir('checkpoints/kd_two_teachers_different_seeds', device=device)
summary = summarize(results)

/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'criterion' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['criterion'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'teacher_ensemble' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['teacher_ensemble'])`.
/home/ivanov.timofey51/kd_vs_hpo/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing

In [117]:
kd_2_exp = results_to_dataframe(summary)
kd_2_exp['teacher_names'] = kd_2_exp.apply(lambda x: func(x), axis=1)

In [132]:
kd_2_res = group_confidence_interval(kd_2_exp[kd_2_exp.role == 'student'].groupby(['arch_idx', 'teacher_names']).test_acc)

In [133]:
base_res

,n,mean,std,sem,ci_low,ci_high,min,max
arch_idx,,,,,,,,
1342,3,0.737800,0.004557,0.002631,0.726479,0.749121,0.7326,0.7411
8712,3,0.805433,0.009419,0.005438,0.782035,0.828832,0.7996,0.8163
11570,3,0.913833,0.002040,0.001178,0.908765,0.918902,0.9116,0.9156


In [134]:
kd1_res

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
1342     11570          3  0.741233  0.004727  0.002729  0.729491  0.752976   
         1342           3  0.750500  0.004397  0.002538  0.739578  0.761422   
8712     11570          3  0.807700  0.007712  0.004452  0.788543  0.826857   
         1342           3  0.803233  0.005169  0.002985  0.790392  0.816075   
11570    11570          3  0.918067  0.002303  0.001330  0.912346  0.923787   
         1342           3  0.875367  0.002854  0.001648  0.868278  0.882456   

                           min     max  
arch_idx teacher_names                  
1342     11570          0.7358  0.7444  
         1342           0.7456  0.7541  
8712     11570          0.8014  0.8163  
         1342           0.8001  0.8092  
11570    11570          0.9157  0.9203  
         1342           0.8732  0.8786

In [135]:
kd_2_res

n      mean       std       sem    ci_low   ci_high  \
arch_idx teacher_names                                                        
8712     11570 11570    3  0.810833  0.008796  0.005078  0.788984  0.832683   
         1342 11570     9  0.810989  0.006782  0.002261  0.805775  0.816202   
         1342 1342      3  0.810767  0.002371  0.001369  0.804876  0.816657   

                           min     max  
arch_idx teacher_names                  
8712     11570 11570    0.8026  0.8201  
         1342 11570     0.8020  0.8226  
         1342 1342      0.8086  0.8133